# DREF Map Data Wrangling
This notebook prepares map-ready data for the 2022 Q1 to 2026 Q1 global allocation map workstream.

It is responsible for filtering the source workbook, deriving quarter fields, recoding modality and pillar fields, enriching with IFRC country API metadata, and exporting stable datasets under the `data` folder.

In [7]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import requests

In [8]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
WORKBOOK_PATH = ROOT / 'DREF_MasterDataset_v1.1 .xlsx'
DATA_DIR = ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

COUNTRY_API_URL = 'https://goadmin.ifrc.org/api/v2/country/'
COUNTRY_API_CACHE = PROCESSED_DIR / 'country_api_reference_raw.json'
START_YEAR = 2022
END_YEAR = 2026

WORKBOOK_PATH

WindowsPath('C:/Users/arun.gandhi/Downloads/DREF_GA_visualizations/DREF_MasterDataset_v1.1 .xlsx')

In [9]:

df = pd.read_excel(WORKBOOK_PATH, sheet_name='ALL_DATA')
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df = df[df['Year'].between(START_YEAR, END_YEAR, inclusive='both')].copy()

df['approval_date'] = pd.to_datetime(df['Date of Approval EnC (start date)'], errors='coerce')

# Drop records where approval date could not be parsed — these cannot be assigned a quarter
missing_dates = df['approval_date'].isna().sum()
if missing_dates > 0:
    print(f"WARNING: {missing_dates} records have no parseable approval date and will be excluded.")
df = df[df['approval_date'].notna()].copy()

df['approval_year'] = df['approval_date'].dt.year
df['approval_quarter'] = df['approval_date'].dt.quarter
df['year_quarter'] = df['approval_year'].astype(str) + ' Q' + df['approval_quarter'].astype(str)
df['quarter_index'] = (df['approval_year'] - START_YEAR) * 4 + df['approval_quarter']
df['total_approved_chf'] = pd.to_numeric(df['Total Approved (CHF)'], errors='coerce')

print(f"Records after filtering: {len(df)}")
print(f"Quarter range: {df['year_quarter'].min()} to {df['year_quarter'].max()}")
df[['Country', 'Region', 'Appeal Type', 'Allocation Type', 'Pillar', 'Disaster Definition', 'year_quarter', 'total_approved_chf']].head()


Records after filtering: 899
Quarter range: 2021 Q1 to 2026 Q2


c:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\.venv\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,Country,Region,Appeal Type,Allocation Type,Pillar,Disaster Definition,year_quarter,total_approved_chf
47,Afghanistan,Asia-Pacific,EA,Loan,Response,Complex Emergency,2022 Q2,750000
63,Brazil,Americas,DREF,Grant,Response,Flood,2022 Q1,81643
81,Democratic Republic of the Congo,Africa,DREF,Grant,Response,Epidemic,2022 Q3,125337
101,Colombia,Americas,DREF,Grant,Response,Flood,2022 Q4,386457
111,Djibouti,Africa,i-DREF,Grant,Anticipatory,Food Insecurity,2022 Q3,400602


In [10]:
def classify_modality(row):
    appeal = str(row['Appeal Type']).strip()
    allocation = str(row['Allocation Type']).strip()
    if allocation == 'Loan':
        return 'Loans'
    if appeal == 'DREF':
        return 'DREF'
    if appeal == 'i-DREF':
        return 'i-DREF'
    if appeal == 's-EAP':
        return 's-EAP'
    if appeal in {'EAP', 'EA'}:
        return 'EAP / EA grants'
    if appeal == 'a-DREF':
        return 'a-DREF'
    return appeal

df['modality_family'] = df.apply(classify_modality, axis=1)
df['pillar_group'] = df['Pillar'].fillna('Unknown').astype(str).str.strip()
df['disaster_definition_clean'] = df['Disaster Definition'].fillna('Unknown').astype(str).str.strip()

top5_disasters = (
    df.groupby('disaster_definition_clean', as_index=False)['total_approved_chf']
      .sum()
      .sort_values('total_approved_chf', ascending=False)
      .head(5)
      .rename(columns={'total_approved_chf': 'total_approved_chf_window'})
)
top5_set = set(top5_disasters['disaster_definition_clean'])
df['disaster_top5_group'] = np.where(df['disaster_definition_clean'].isin(top5_set), df['disaster_definition_clean'], 'Other')

top5_disasters

,disaster_definition_clean,total_approved_chf_window
9,Flood,102993338
7,Epidemic,40131713
15,Population Movement,29477771
4,Cyclone,29428159
3,Complex Emergency,20958730


In [11]:

def fetch_country_api(base_url, page_size=200):
    """Fetch all pages from the paginated IFRC GO country API.

    The GO API uses DRF offset/limit pagination (default 50 records/page).
    We request page_size=200 to minimise round-trips while staying within
    the API's supported limit. The loop follows the 'next' URL until None.
    A count assertion validates that no pages were missed.
    """
    rows = []
    # Request a large page size up-front to reduce API round-trips
    next_url = f"{base_url}?limit={page_size}"
    session = requests.Session()
    total_expected = None
    while next_url:
        response = session.get(next_url, timeout=60)
        response.raise_for_status()
        payload = response.json()
        if total_expected is None:
            total_expected = payload.get('count', 0)
        rows.extend(payload.get('results', []))
        next_url = payload.get('next')
        print(f"  Fetched {len(rows)} / {total_expected} records...")
    # Validate completeness
    if total_expected is not None and len(rows) != total_expected:
        raise ValueError(
            f"Pagination mismatch: expected {total_expected} records, got {len(rows)}. "
            "Check if the API added or removed records mid-fetch."
        )
    print(f"API fetch complete: {len(rows)} total records.")
    return rows

if COUNTRY_API_CACHE.exists():
    print(f"Loading from cache: {COUNTRY_API_CACHE}")
    country_rows = json.loads(COUNTRY_API_CACHE.read_text(encoding='utf-8'))
    print(f"Cache contains {len(country_rows)} records.")
else:
    print("Cache not found — fetching from IFRC GO API...")
    country_rows = fetch_country_api(COUNTRY_API_URL)
    COUNTRY_API_CACHE.write_text(json.dumps(country_rows, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f"Cache written to {COUNTRY_API_CACHE}")

country_api = pd.DataFrame(country_rows)
country_api = country_api[country_api['record_type'] == 1].copy()
country_api['api_country_name'] = country_api['name'].astype(str).str.strip()
country_api['iso3'] = country_api['iso3'].fillna('').astype(str).str.strip()
country_api['longitude'] = country_api['centroid'].apply(
    lambda value: value.get('coordinates', [None, None])[0] if isinstance(value, dict) else None
)
country_api['latitude'] = country_api['centroid'].apply(
    lambda value: value.get('coordinates', [None, None])[1] if isinstance(value, dict) else None
)

print(f"\nrecord_type=1 countries: {len(country_api)}")
country_api[['api_country_name', 'iso3', 'longitude', 'latitude']].head()


Loading from cache: C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\data\processed\country_api_reference_raw.json
Cache contains 281 records.

record_type=1 countries: 253


,api_country_name,iso3,longitude,latitude
0,Afghanistan,AFG,67.709953,33.939110
2,Aksai Chin,,NaN,NaN
3,Albania,ALB,20.168331,41.153332
4,Algeria,DZA,1.659626,28.033886
5,American Samoa,ASM,-170.132217,-14.270972


In [12]:

# Country name overrides: maps workbook spelling → IFRC GO API name.
# Root causes fall into 4 categories:
#   1. Formal vs short name  (Iran, Syria, Russia, Tanzania, Gambia, Eswatini, Lao PDR, Micronesia)
#   2. Hyphenation/spacing   (Guinea Bissau, Timor Leste, Republic of Congo)
#   3. Renamed country       (Turkey → Türkiye 2022, Cape Verde → Cabo Verde 2013)
#   4. Different convention  (Democratic Republic of the Congo → without "the", Vietnam → Viet Nam, Czechia → Czech Republic)
#
# Secretariats ('Africa Secretariat', 'America Secretariat', 'MENA Secretariat') are
# IFRC regional offices, not countries — they are intentionally excluded from maps.

COUNTRY_NAME_OVERRIDES = {
    # Formal names
    'Iran':                                 'Iran, Islamic Republic of',
    'Syria':                                'Syrian Arab Republic',
    'Russia':                               'Russian Federation',
    'Tanzania':                             'Tanzania, United Republic of',
    'Gambia':                               'Gambia, Republic of The',
    'Eswatini':                             'Eswatini, Kingdom of',
    'Lao PDR':                              "Lao People's Democratic Republic",
    'Micronesia':                           'Micronesia, Federated States of',
    # Hyphenation / spacing
    'Guinea Bissau':                        'Guinea-Bissau',
    'Timor Leste':                          'Timor-Leste',
    'Republic of Congo':                    'Congo',
    # Renamed countries
    'Turkey':                               'Türkiye',
    'Cape Verde':                           'Cabo Verde',
    # Different convention
    'Democratic Republic of the Congo':     'Democratic Republic of Congo',
    'Vietnam':                              'Viet Nam',
    'Czechia':                              'Czech Republic',
}

# Entries that should be excluded from map joins (regional secretariats, not countries)
NON_COUNTRY_ENTRIES = {'Africa Secretariat', 'America Secretariat', 'MENA Secretariat'}

df['country_clean'] = df['Country'].astype(str).str.strip().replace(COUNTRY_NAME_OVERRIDES)
country_lookup = country_api[['api_country_name', 'iso3', 'longitude', 'latitude']].drop_duplicates()
map_df = df.merge(country_lookup, left_on='country_clean', right_on='api_country_name', how='left')

join_audit = (
    map_df[['Country', 'country_clean', 'iso3', 'longitude', 'latitude']]
      .drop_duplicates()
      .assign(join_status=lambda frame: np.where(
          frame['iso3'].notna() & frame['iso3'].ne(''), 'matched', 'unmatched'))
      .sort_values(['join_status', 'Country'])
)

# Rank countries by total CHF regardless of coordinate availability,
# then join coordinates separately — prevents high-CHF countries from being
# silently excluded just because their name join failed.
top10_by_chf = (
    map_df[~map_df['Country'].isin(NON_COUNTRY_ENTRIES)]
      .groupby('country_clean', as_index=False)['total_approved_chf']
      .sum()
      .sort_values('total_approved_chf', ascending=False)
      .head(10)
)
top10_countries = top10_by_chf.merge(
    country_lookup[['api_country_name', 'iso3', 'longitude', 'latitude']].drop_duplicates(),
    left_on='country_clean',
    right_on='api_country_name',
    how='left'
).rename(columns={'total_approved_chf': 'total_approved_chf_window'})
top10_set = set(top10_countries['country_clean'])

# Group by semantic dimensions only — coordinates re-joined after aggregation
country_quarter = (
    map_df[map_df['country_clean'].isin(top10_set)]
      .groupby(
          ['year_quarter', 'quarter_index', 'country_clean', 'Region',
           'modality_family', 'pillar_group', 'disaster_top5_group'],
          as_index=False
      )['total_approved_chf']
      .sum()
)
country_quarter = country_quarter.merge(
    country_lookup[['api_country_name', 'iso3', 'longitude', 'latitude']].drop_duplicates(),
    left_on='country_clean',
    right_on='api_country_name',
    how='left'
)

region_quarter = (
    map_df[~map_df['Country'].isin(NON_COUNTRY_ENTRIES)]
      .groupby(
          ['year_quarter', 'quarter_index', 'Region',
           'modality_family', 'pillar_group', 'disaster_top5_group'],
          as_index=False
      )['total_approved_chf']
      .sum()
)

country_api.to_csv(PROCESSED_DIR / 'country_api_reference.csv', index=False)
join_audit.to_csv(PROCESSED_DIR / 'country_name_join_audit.csv', index=False)
map_df.to_csv(PROCESSED_DIR / 'dref_map_base_2022q1_2026q1.csv', index=False)
country_quarter.to_csv(PROCESSED_DIR / 'dref_map_country_quarter.csv', index=False)
region_quarter.to_csv(PROCESSED_DIR / 'dref_map_region_quarter.csv', index=False)
top10_countries.to_csv(PROCESSED_DIR / 'dref_map_top10_countries.csv', index=False)
top5_disasters.to_csv(PROCESSED_DIR / 'dref_map_top5_disasters.csv', index=False)

unmatched = join_audit[
    (join_audit['join_status'] == 'unmatched') &
    (~join_audit['Country'].isin(NON_COUNTRY_ENTRIES))
]
print(f"Processed map data written to {PROCESSED_DIR}")
print(f"Top 10 countries: {sorted(top10_set)}")
print(f"\nNon-country entries excluded from maps: {sorted(NON_COUNTRY_ENTRIES)}")
print(f"\nStill unmatched real countries ({len(unmatched)}): {list(unmatched['Country'].unique())}")


Processed map data written to C:\Users\arun.gandhi\Downloads\DREF_GA_visualizations\data\processed
Top 10 countries: ['Afghanistan', 'Bangladesh', 'Democratic Republic of Congo', 'Ethiopia', 'Iran, Islamic Republic of', 'Kenya', 'Lebanon', 'Philippines', 'Somalia', 'Syrian Arab Republic']

Non-country entries excluded from maps: ['Africa Secretariat', 'America Secretariat', 'MENA Secretariat']

Still unmatched real countries (0): []
